# Visualização da correção do ERA5 (`artifacts/corrected_grid/v1/`)

Compara, para cada dia e estação INMET selecionados:
- **ERA5 original** (`ws_max` reconstruído como `rajada_max_corrigida - bias`)
- **ERA5 corrigido** (`rajada_max_corrigida` — IDW do resíduo do modelo vencedor por cluster/trimestre, ver `src/dataset/creation/grid_generator.py`)
- **Observado INMET** (`daily_wind_gust_max`, rajada máxima diária, mesma grandeza física da correção)

Não é preciso abrir o ERA5-Basin (`ERA5_Features_Basin_2000_2026.nc`, ~6GB): como `rajada_corrigida = era5_original + bias` (ver linha `rajada_corrigida = np.clip(era5_vals + bias_grid, 0, None)`), o campo original é recuperado localmente a partir do próprio grid corrigido.

In [7]:
%matplotlib inline

from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from ipywidgets import interact

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CORRECTED_GRID_DIR = REPO_ROOT / "artifacts" / "corrected_grid" / "v1"
INMET_PATH = REPO_ROOT / "dataset" / "raw" / "INMET_Stratified.nc"

In [8]:
# Grid corrigido: concatena todos os anos disponíveis em artifacts/corrected_grid/v1/
grid_files = sorted(CORRECTED_GRID_DIR.glob("grid_corrected_*.nc"))
assert grid_files, f"Nenhum grid_corrected_*.nc encontrado em {CORRECTED_GRID_DIR}"

ds_era = xr.open_mfdataset(grid_files, combine="by_coords", chunks={"time": 50})
ds_era = ds_era.sortby("time")

# ERA5 original recuperado localmente: rajada_corrigida = era5_original + bias
ds_era["ws_original"] = ds_era["rajada_max_corrigida"] - ds_era["bias"]

lat_min, lat_max = float(ds_era.latitude.min()), float(ds_era.latitude.max())
lon_min, lon_max = float(ds_era.longitude.min()), float(ds_era.longitude.max())

print(ds_era)

<xarray.Dataset> Size: 193MB
Dimensions:               (time: 1820, latitude: 86, longitude: 77)
Coordinates:
  * time                  (time) datetime64[ns] 15kB 2020-01-08 ... 2024-12-31
    in_sample             (time) bool 2kB dask.array<chunksize=(50,), meta=np.ndarray>
  * latitude              (latitude) float64 688B -13.75 -14.0 ... -34.75 -35.0
  * longitude             (longitude) float64 616B -58.0 -57.75 ... -39.25 -39.0
Data variables:
    rajada_max_corrigida  (time, latitude, longitude) float32 48MB dask.array<chunksize=(50, 86, 77), meta=np.ndarray>
    direcao_vento         (time, latitude, longitude) float32 48MB dask.array<chunksize=(50, 86, 77), meta=np.ndarray>
    bias                  (time, latitude, longitude) float32 48MB dask.array<chunksize=(50, 86, 77), meta=np.ndarray>
    ws_original           (time, latitude, longitude) float32 48MB dask.array<chunksize=(50, 86, 77), meta=np.ndarray>
Attributes:
    source:                IRC Vendaval — corrected_grid (b

/tmp/ipykernel_291678/453753286.py:5: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 50. This could degrade performance. Instead, consider rechunking after loading.
  ds_era = xr.open_mfdataset(grid_files, combine="by_coords", chunks={"time": 50})
/tmp/ipykernel_291678/453753286.py:5: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 50. This could degrade performance. Instead, consider rechunking after loading.
  ds_era = xr.open_mfdataset(grid_files, combine="by_coords", chunks={"time": 50})
/tmp/ipykernel_291678/453753286.py:5: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 50. This could degrade performance. Instead, consider rechunking after loading.
  ds_era = xr.open_mfdataset(grid_files, combine="by_coords", chunks={"time": 50})
/tmp/ipykernel_291678/453753286.py:5: UserWarning: The specified chunks separate the stored

In [9]:
# Observações INMET (rajada máxima diária — mesma grandeza física da correção)
ds_inmet = xr.open_dataset(INMET_PATH)

stations_list = sorted(ds_inmet["estacao"].values.tolist())
print(f"{len(stations_list)} estações INMET disponíveis")

243 estações INMET disponíveis


In [10]:
# Série temporal média na bacia (só como referência de contexto — ver nota abaixo)
era5_ts_basin = ds_era["ws_original"].mean(dim=["latitude", "longitude"]).compute()
era5_corrected_ts_basin = ds_era["rajada_max_corrigida"].mean(dim=["latitude", "longitude"]).compute()


def nearest_gridpoint_series(var: str, lat: float, lon: float):
    """Série no pixel do grid mais próximo de (lat, lon) — comparação correta
    ponto-a-ponto com uma estação, em vez de média espacial da bacia inteira
    (rajada é um fenômeno local; a média da bacia suaviza os picos)."""
    return ds_era[var].sel(latitude=lat, longitude=lon, method="nearest").compute()

In [ ]:
def update_plots(station_name, map_date):
    try:
        ds_single_inmet = ds_inmet.sel(estacao=station_name)
        inmet_ts = ds_single_inmet["daily_wind_gust_max"]

        fig = plt.figure(figsize=(14, 11))
        gs = fig.add_gridspec(2, 2, height_ratios=[2, 1], width_ratios=[1, 1])

        # ========== Mapa 1: ERA5 original ==========
        ax1 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
        ax1.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        wind_orig = ds_era["ws_original"].sel(time=str(map_date), method="nearest").squeeze()

        vmax_scale = 20.0  # rajada máxima (m/s) — ajuste conforme a data escolhida

        plot1 = ax1.pcolormesh(ds_era.longitude, ds_era.latitude, wind_orig,
                               transform=ccrs.PlateCarree(), cmap="viridis", shading="auto",
                               vmin=0, vmax=vmax_scale)
        plt.colorbar(plot1, ax=ax1, orientation="horizontal", fraction=0.04, pad=0.04,
                     label="Rajada máx. (m/s) - ERA5 original")

        ax1.scatter(ds_inmet.longitude, ds_inmet.latitude, color="red", marker="^",
                    edgecolor="black", s=50, transform=ccrs.PlateCarree(), label="Estações INMET", zorder=5)
        ax1.scatter(ds_single_inmet.longitude, ds_single_inmet.latitude, color="yellow", marker="^",
                    edgecolor="black", s=150, transform=ccrs.PlateCarree(), label=f"Selecionada: {station_name}", zorder=6)
        ax1.add_feature(cfeature.COASTLINE, linewidth=1.2)
        ax1.add_feature(cfeature.BORDERS, linestyle=":", linewidth=1)
        ax1.add_feature(cfeature.STATES, linestyle=":", linewidth=0.5)
        ax1.legend(loc="lower right")
        ax1.set_title("ERA5 original", fontsize=12)

        # ========== Mapa 2: ERA5 corrigido ==========
        ax2 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())
        ax2.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        wind_corr = ds_era["rajada_max_corrigida"].sel(time=str(map_date), method="nearest").squeeze()

        plot2 = ax2.pcolormesh(ds_era.longitude, ds_era.latitude, wind_corr,
                               transform=ccrs.PlateCarree(), cmap="viridis", shading="auto",
                               vmin=0, vmax=vmax_scale)
        plt.colorbar(plot2, ax=ax2, orientation="horizontal", fraction=0.04, pad=0.04,
                     label="Rajada máx. (m/s) - ERA5 corrigido")

        ax2.scatter(ds_inmet.longitude, ds_inmet.latitude, color="red", marker="^",
                    edgecolor="black", s=50, transform=ccrs.PlateCarree(), zorder=5)
        ax2.scatter(ds_single_inmet.longitude, ds_single_inmet.latitude, color="yellow", marker="^",
                    edgecolor="black", s=150, transform=ccrs.PlateCarree(), zorder=6)
        ax2.add_feature(cfeature.COASTLINE, linewidth=1.2)
        ax2.add_feature(cfeature.BORDERS, linestyle=":", linewidth=1)
        ax2.add_feature(cfeature.STATES, linestyle=":", linewidth=0.5)
        ax2.set_title("ERA5 corrigido", fontsize=12)

        # ========== Gráfico de série temporal ==========
        ax3 = fig.add_subplot(gs[1, :])

        ts_year = str(map_date).split("-")[0]
        ts_start = f"{ts_year}-01-01"
        ts_end = f"{ts_year}-12-31"

        era5_ts_aligned = era5_ts.sel(time=slice(ts_start, ts_end))
        era5_corr_ts_aligned = era5_corrected_ts.sel(time=slice(ts_start, ts_end))
        inmet_ts_aligned = inmet_ts.sel(time=slice(ts_start, ts_end))

        ax3.plot(era5_ts_aligned.time, era5_ts_aligned,
                 label="ERA5", color="#1f77b4", linewidth=1.5, alpha=0.8)
        ax3.plot(era5_corr_ts_aligned.time, era5_corr_ts_aligned,
                 label="Base V1", color="#2ca02c", linewidth=1.5, alpha=0.8, linestyle="--")
        ax3.plot(inmet_ts_aligned.time, inmet_ts_aligned,
                 label=f"INMET - {station_name}", color="#d62728", linewidth=1.5, alpha=0.8)

        ax3.set_xlabel("Tempo", fontsize=11)
        ax3.set_ylabel("Rajada máxima (m/s)", fontsize=11)
        ax3.legend(loc="upper right")
        ax3.grid(True, linestyle="--", alpha=0.5)

        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"ERRO AO PLOTAR: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
# =============================================================================
# EXECUTAR O INTERACT (Garanta que esta célula seja executada!)
# =============================================================================
interact(
    update_plots,
    station_name=widgets.Dropdown(options=stations_list, value=stations_list[0], description="Estação:"),
    map_date=widgets.Text(value="2020-05-15", description="Data Mapa:"),
)

interactive(children=(Dropdown(description='Estação:', options=('A502', 'A505', 'A506', 'A507', 'A508', 'A509'…

<function __main__.update_plots(station_name, map_date)>